<a href="https://colab.research.google.com/github/simecek/dspracticum2026/blob/main/lesson02/05_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 5: Fine-tuning a pretrained model

1. Python basics & the training loop
2. Dense neural network on FashionMNIST
3. Convolutional neural network (CNN) on FashionMNIST
4. The same with fastai
5. **Fine-tuning a pretrained model** ← *you are here*

So far we have always started from a network with **random** weights. In practice, almost nobody does that. Instead, we take a big network that someone has already trained on millions of photos, and **fine-tune** it for our task. This is called **transfer learning**.

**What you will learn here:**
- what a pretrained model already knows
- how to fine-tune it to recognize **37 breeds of cats and dogs** in a few minutes
- what `fine_tune` does inside (*freezing* and *unfreezing*)
- **data augmentation**, and how to try the model on your own photo

**Before you start:** *Runtime → Change runtime type → T4 GPU*. This time the GPU is a must.

In [ ]:
from fastai.vision.all import *

try:
    import timm
except ImportError:
    !pip install -q timm
    import timm

---
## 1. The idea

Remember notebook 3: the first CNN layer learned to detect **edges**, and the second layer combined them into more complex patterns. A big network trained on millions of photos has learned a whole hierarchy of such patterns: edges → textures → fur, eyes, ears → faces, bodies...

These patterns are useful for almost any image task. So we:
1. take the pretrained network and **keep** everything it learned (the *body*)
2. **replace** only its last layer (the *head*), which decides between its original categories, with a new one for **our** categories
3. **train** a little, mostly the new head

It's like hiring someone who already knows how to see. You only need to teach them the names of the breeds.

---
## 2. The data: Oxford-IIIT Pets

The [Oxford-IIIT Pet dataset](https://www.robots.ox.ac.uk/~vgg/data/pets/) has about 7,400 photos of 37 breeds: 12 cat breeds and 25 dog breeds. fastai can download it for us:

In [ ]:
path = untar_data(URLs.PETS) / "images"
files = get_image_files(path).sorted()
print("number of photos:", len(files))
files[:5]

This time there are no class folders: the **breed is in the file name**. (Fun fact: cat breeds start with a capital letter, dog breeds with a lowercase one.) A small function extracts it:

In [ ]:
def get_breed(name):
    return name.rsplit("_", 1)[0]     # "great_pyrenees_173.jpg" -> "great_pyrenees"

print(files[0].name, "->", get_breed(files[0].name))

Unlike FashionMNIST, these are real color photos of **different sizes**:

In [ ]:
for file in files[:3]:
    print(file.name, PILImage.create(file).size)

A network needs all images in a batch to have the same size, so we resize them to 224×224 pixels (the size the pretrained model was trained on). And since there is no separate test set, fastai randomly holds out 20% of the photos as the **validation set**.

### Data augmentation

With only ~150 photos per breed, the model could simply memorize them. **Data augmentation** creates slightly different versions of each photo every time it is used: flipped, rotated, zoomed, brighter or darker. The model never sees exactly the same image twice, which reduces overfitting.

In [ ]:
dls = ImageDataLoaders.from_name_func(path, files, get_breed, valid_pct=0.2, seed=42,
                                      item_tfms=Resize(224),        # same size for every image
                                      batch_tfms=aug_transforms())  # data augmentation
print("training photos:  ", len(dls.train_ds))
print("validation photos:", len(dls.valid_ds))
dls.show_batch(max_n=8)

### Look inside: augmentation

The same photo, as the model sees it in 8 different batches:

In [ ]:
dls.train.show_batch(max_n=8, unique=True)

---
## 3. What does a pretrained model know?

We will use **ConvNeXt-tiny**, a modern CNN, trained on over 14 million photos from [ImageNet](https://www.image-net.org/) to recognize 1,000 categories: animals, vehicles, food, tools... The [timm](https://huggingface.co/timm) library offers hundreds of such pretrained models.

In [ ]:
model_name = "convnext_tiny.fb_in22k_ft_in1k"
pretrained = timm.create_model(model_name, pretrained=True).eval()

print(f"number of parameters: {sum(p.numel() for p in pretrained.parameters()):,}")
print("the last layer (head):", pretrained.get_classifier())

28 million parameters, and the last layer has **1,000 outputs**, one per ImageNet category. Let's ask the model about two photos from our validation set. Run the helper cell first:

In [ ]:
# @title Helper: ask the original ImageNet model (just run this cell)
transform = timm.data.create_transform(**timm.data.resolve_data_config({}, model=pretrained))
imagenet = timm.data.ImageNetInfo()

def imagenet_top3(file):
    image = PILImage.create(file)
    with torch.no_grad():
        probabilities = pretrained(transform(image).unsqueeze(0)).softmax(dim=1)[0]
    image.show(figsize=(3, 3), title=f"true breed: {get_breed(file.name)}")
    plt.show()
    top = probabilities.topk(3)
    for p, i in zip(top.values, top.indices):
        print(f"{imagenet.index_to_description(i.item()):40s} {p:.1%}")

In [ ]:
valid_files = sorted(dls.valid_ds.items)
dog = [f for f in valid_files if get_breed(f.name) == "beagle"][0]
cat = [f for f in valid_files if get_breed(f.name) == "Bengal"][0]

imagenet_top3(dog)

In [ ]:
imagenet_top3(cat)

ImageNet happens to contain *beagle* as a category, so the model already knows it. But there is **no Bengal cat** among its 1,000 categories, so it answers with something similar. The model clearly *sees* the cat well, it just doesn't know our labels. Fine-tuning fixes exactly that.

---
## 4. Fine-tuning

`vision_learner` does steps 1 and 2 of our plan: it downloads the pretrained model, cuts off its head and attaches a new, randomly initialized head with **37 outputs**, one per breed.

In [ ]:
learn = vision_learner(dls, model_name, metrics=accuracy)

### Look inside: body and head

`learn.model` has two parts: `learn.model[0]` is the pretrained body, `learn.model[1]` is the new head. Note the last layer with 37 outputs:

In [ ]:
learn.model[1]

### Freezing

At first, the body is **frozen**: its weights are not updated, only the new head learns. Why? The new head is random, and its first gradients are basically noise. We don't want that noise to damage the carefully pretrained body.

In [ ]:
def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

total = sum(p.numel() for p in learn.model.parameters())
print(f"total parameters:     {total:,}")
print(f"trainable (frozen):   {count_trainable(learn.model):,}")

Only a small fraction of the parameters will be trained at first (the new head and a few normalization layers).

As in notebook 4, let's find a good learning rate:

In [ ]:
suggestion = learn.lr_find()
suggestion

### `fine_tune`: freeze, then unfreeze

`fine_tune(3)` does two phases:
1. **1 epoch with the body frozen**: the new head learns the basics
2. **unfreeze** everything and train **3 more epochs**, so the whole network adapts a little to our photos (the early layers with a smaller learning rate, as they already know useful basics like edges)

On the Colab GPU this takes a few minutes:

In [ ]:
learn.fine_tune(3, suggestion.valley)

In [ ]:
print(f"trainable (unfrozen): {count_trainable(learn.model):,}")

### Look inside: what `fine_tune` does

`fine_tune` is just a shortcut for things we already know from notebook 4:

```python
learn.freeze()                        # train only the head
learn.fit_one_cycle(1, lr)
learn.unfreeze()                      # now train everything
learn.fit_one_cycle(3, slice(lr / 200, lr / 2))   # smaller learning rates for the early layers
```

Here is the learning rate schedule of the last phase (the 3 unfrozen epochs), the familiar one-cycle shape:

In [ ]:
learn.recorder.plot_sched()

About **95% accuracy on 37 breeds**, after a few minutes of training on ~6,000 photos! Training such a network from random weights would need far more data and time (and would still be worse).

---
## 5. Results

The same two photos as before, now with the fine-tuned model:

In [ ]:
def predict_breed(image):
    breed, index, probabilities = learn.predict(image)
    top = probabilities.topk(3)
    for p, i in zip(top.values, top.indices):
        print(f"{dls.vocab[int(i)]:28s} {p:.1%}")

for file in [dog, cat]:
    PILImage.create(file).show(figsize=(3, 3), title=f"true breed: {get_breed(file.name)}")
    plt.show()
    predict_breed(PILImage.create(file))

With 37 classes, the confusion matrix would be too big to read, so let's list the most common mistakes, as (true breed, predicted breed, count):

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
interp.most_confused(min_val=3)

**Try it:** Search the web for photos of these breeds. Would you tell them apart?

And the photos with the highest loss:

In [ ]:
interp.plot_top_losses(6, nrows=2, figsize=(14, 8))

---
## 6. Try your own photo

Upload a photo of a cat or a dog (works in Colab). What does the model say about a breed it has never seen, or about a photo of yourself?

In [ ]:
from google.colab import files as colab_files

uploaded = colab_files.upload()
image = PILImage.create(list(uploaded.keys())[0])
image.show(figsize=(4, 4))
plt.show()
predict_breed(image)

### Saving the model

`export` saves the trained model into a single file. You can download it from the Colab file browser (folder icon on the left) and use it later, e.g. in a web app.

In [ ]:
learn.export("pets_model.pkl")

---
## 7. Summary

| | notebooks 2-4 | this notebook |
|---|---|---|
| starting weights | random | **pretrained** on 14 million photos |
| data | 60,000 tiny grayscale images | ~6,000 color photos |
| model | our CNN, 0.4 million parameters | ConvNeXt-tiny, 28 million parameters |
| training | `fit_one_cycle` | `fine_tune`: head first (frozen), then everything |
| accuracy | ~92% on 10 classes | ~95% on 37 breeds |

**Transfer learning** is how most image models are built in practice: start from a pretrained model and fine-tune it on your own (often small) dataset.

### Exercises
1. Try `fine_tune(1)` and `fine_tune(6)`. How much do extra epochs help?
2. Try another pretrained model, e.g. `"resnet18"` (smaller and faster) or `"convnext_small.fb_in22k_ft_in1k"` (bigger). You can list more with `timm.list_models("convnext*", pretrained=True)`.
3. Train **without** pretraining: `learn = vision_learner(dls, model_name, pretrained=False, metrics=accuracy)`, then `learn.fit_one_cycle(4, 1e-3)`. How good is it in the same time?
4. Change the data augmentation, e.g. `aug_transforms(max_rotate=30, max_zoom=1.5, max_lighting=0.4)`, and look at `dls.train.show_batch(max_n=8, unique=True)`. Then remove it completely (delete `batch_tfms=...`) and fine-tune again. Does the validation accuracy change?
5. **Homework preview:** how would you fine-tune this model on *your own* image dataset? (Hint: notebook 4, `ImageDataLoaders.from_folder`.)